# Baselines - week 3
Shireen

Goal this week is the first GNN + comparing it to the baselines, so I updated the baselines to work on the same data the GNN will use.

What changed from week 2:
- everything gets converted to an AIG with ABC (`strash`) first, and the graph is built the way the GNN deck shows it: PI / AND / PO nodes, and NOTs become an `inverted` flag on the edge instead of their own node. I used the same attribute names as Pranav's Chunk1 (`gate`, `inverted`) so the loader can be reused
- added a 3rd baseline, the structural one from the GNN slides: `[#nodes, #edges, depth, #PI, #PO]`
- added `embedding_ranker()` so we can drop the GNN's embeddings in and score it the same way as the baselines

Each baseline still has the 2 functions (one that computes it, one that takes a target + list of circuits and returns them ranked)

## setup

In [1]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y yosys > /dev/null
!pip -q install grakel
!which yosys-abc

W: Failed to fetch https://download.docker.com/linux/ubuntu/dists/noble/InRelease  Invalid response from proxy: HTTP/1.1 403 Forbidden  Content-Type: text/plain; charset=utf-8  X-Content-Type-Options: nosniff  Content-Length: 77  Connection: close     [IP: 127.0.0.1 37339]
W: Some index files failed to download. They have been ignored, or old ones used instead.


/usr/bin/yosys-abc


In [2]:
import os, re, glob, shutil, subprocess
import numpy as np
import pandas as pd
import networkx as nx

## circuits
Grabbing the 11 originals from Pranav's branch so I don't have to upload anything.

Variants go in `variants/` named like `c432_v3.bench`. Whatever's before the first `_` is the circuit it came from, that's how the eval knows which ones should match.

In [3]:
!rm -rf repo && git clone -q --depth 1 -b PranavPrasanna https://github.com/acm-research-f26/Circuitree repo
ORIG_DIR = "originals"
os.makedirs(ORIG_DIR, exist_ok=True)
for f in glob.glob("repo/c*.bench"):
    shutil.copy(f, ORIG_DIR)
print(sorted(os.listdir(ORIG_DIR)))

VARIANT_DIR = "variants"          # put the team's variant .bench files here
os.makedirs(VARIANT_DIR, exist_ok=True)
USE_STANDIN_VARIANTS = True       # set to False once the team's variants are in VARIANT_DIR

['c1355.bench', 'c17.bench', 'c1908.bench', 'c2670.bench', 'c3540.bench', 'c432.bench', 'c499.bench', 'c5315.bench', 'c6288.bench', 'c7552.bench', 'c880.bench']


## convert to AIG

In [4]:
import os, re, glob, shutil, subprocess


def make_aigs(src_dir, out_dir="aig"):
    # run ABC `strash` on every .bench in src_dir and save the AIG to out_dir
    shutil.rmtree(out_dir, ignore_errors=True)      # start clean so old files don't sneak in
    os.makedirs(out_dir)
    paths = sorted(glob.glob(os.path.join(src_dir, "**", "*.bench"), recursive=True))
    for p in paths:
        out = os.path.join(out_dir, os.path.basename(p))
        r = subprocess.run(["yosys-abc", "-c", f'read_bench "{p}"; strash; write_bench -l "{out}"'],
                           capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists(out):
            print("ABC failed on", p, r.stdout[-300:], r.stderr[-300:])
    print(f"{len(paths)} circuits -> {out_dir}/")
    return out_dir

## load AIG -> graph
NOTs get folded into the edge (inverted = True). Checked this by simulating the graphs on random inputs vs the original circuits and they all matched

In [5]:
import networkx as nx


class NotAIGError(ValueError):
    pass


_GATE = re.compile(r"^\s*(\S+)\s*=\s*([A-Za-z_]\w*)\s*(0x[0-9a-fA-F]+)?\s*\((.*)\)\s*$")
_IO = re.compile(r"^\s*(INPUT|OUTPUT)\s*\(\s*([^\s\)]+)\s*\)\s*$", re.I)
_CONST = re.compile(r"^\s*(\S+)\s*=\s*(vdd|gnd|CONST0|CONST1|0|1)\s*(\(\s*\))?\s*$", re.I)


def load_aig(path):
    # read an AIG .bench (ABC `write_bench -l` output) into a networkx DiGraph
    defs, pis, pos = {}, [], []
    with open(path) as f:
        for raw in f:
            line = raw.split("#", 1)[0].strip()
            if not line:
                continue
            m = _IO.match(line)
            if m:
                (pis if m.group(1).upper() == "INPUT" else pos).append(m.group(2))
                continue
            m = _CONST.match(line)
            if m:
                defs[m.group(1)] = ("CONST", [], "1" if m.group(2).lower() in ("vdd", "const1", "1") else "0")
                continue
            m = _GATE.match(line)
            if m:
                out, op, hexv, args = m.groups()
                defs[out] = (op.upper(), [a.strip() for a in args.split(",") if a.strip()], hexv)

    G = nx.DiGraph()
    for p in pis:
        G.add_node(p, gate="INPUT")
    lit = {p: (p, False) for p in pis}          # signal -> (driving node, inverted?)

    def make_and(name, fanins, invs, out_inv=False):
        G.add_node(name, gate="AND")
        for (node, inv), extra in zip(fanins, invs):
            G.add_edge(node, name, inverted=inv ^ extra)
        return (name, out_inv)

    def build(s):
        op, args, hexv = defs[s]
        ins = [lit[a] for a in args]
        if op == "CONST":
            G.add_node("CONST0", gate="CONST")
            return ("CONST0", hexv == "1")
        if op == "NOT":
            return (ins[0][0], not ins[0][1])
        if op in ("BUF", "BUFF"):
            return ins[0]
        if op == "AND":
            return make_and(s, ins, [False] * len(ins))
        if op == "LUT":                        # write_bench without -l
            tt, k = int(hexv, 16), len(ins)
            if k == 1:
                return (ins[0][0], ins[0][1] ^ bool(tt & 1))
            bits = [(tt >> i) & 1 for i in range(1 << k)]
            for want, out_inv in ((1, False), (0, True)):
                hits = [i for i, b in enumerate(bits) if b == want]
                if len(hits) == 1:             # AND of (possibly inverted) inputs
                    return make_and(s, ins, [not (hits[0] >> j) & 1 for j in range(k)], out_inv)
        raise NotAIGError(f"{os.path.basename(path)}: '{s} = {op}(...)' is not AIG. "
                          "Run make_aigs() (ABC strash) on it first.")

    def resolve(sig):                          # iterative, so deep circuits don't hit recursion limits
        stack = [sig]
        while stack:
            s = stack[-1]
            if s in lit:
                stack.pop(); continue
            if s not in defs:
                raise ValueError(f"{os.path.basename(path)}: signal '{s}' is used but never defined")
            todo = [a for a in defs[s][1] if a not in lit]
            if todo:
                stack.extend(todo); continue
            stack.pop()
            lit[s] = build(s)
        return lit[sig]

    for i, po in enumerate(pos):
        node, inv = resolve(po)
        G.add_node(f"PO{i}:{po}", gate="OUTPUT")
        G.add_edge(node, f"PO{i}:{po}", inverted=inv)
    return G


def load_folder(folder):
    # load every .bench in a folder -> {circuit_name: graph}
    paths = sorted(glob.glob(os.path.join(folder, "**", "*.bench"), recursive=True))
    return {os.path.splitext(os.path.basename(p))[0]: load_aig(p) for p in paths}


def aig_stats(G):
    # [#nodes, #edges, depth, #PI, #PO]
    level = {}
    for n in nx.topological_sort(G):
        preds = list(G.predecessors(n))
        base = max((level[p] for p in preds), default=0)
        level[n] = base + (1 if G.nodes[n]["gate"] == "AND" else 0)
    gates = [d["gate"] for _, d in G.nodes(data=True)]
    return {"nodes": G.number_of_nodes(), "edges": G.number_of_edges(),
            "depth": max(level.values(), default=0),
            "PI": gates.count("INPUT"), "PO": gates.count("OUTPUT")}

## baseline 1 - histogram
same idea as last week but on the AIG: proportions of AND / PI / PO / normal edges / inverted edges

In [6]:
import numpy as np


def _cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else 0.0


def histogram_vector(G):
    # function 1: normalised counts of [AND, PI, PO, normal edges, inverted edges]
    gates = [d["gate"] for _, d in G.nodes(data=True)]
    inv = sum(1 for *_, d in G.edges(data=True) if d.get("inverted"))
    v = np.array([gates.count("AND"), gates.count("INPUT"), gates.count("OUTPUT"),
                  G.number_of_edges() - inv, inv], dtype=float)
    return v / v.sum() if v.sum() else v


def histogram_rank(target, candidates, circuits):
    # function 2: [(name, similarity), ...] most similar first (cosine)
    t = histogram_vector(circuits[target])
    out = [(c, _cosine(t, histogram_vector(circuits[c]))) for c in candidates if c != target]
    return sorted(out, key=lambda x: x[1], reverse=True)

## baseline 2 - WL kernel
grakel only does node labels so I put an INV node in the middle of each inverted edge

In [7]:
def _wl_ready(G):
    # grakel only understands node labels, so each inverted edge gets an 'INV' node in the middle
    H = nx.Graph()
    for n, d in G.nodes(data=True):
        H.add_node(n, gate=d["gate"])
    for u, v, d in G.edges(data=True):
        if d.get("inverted"):
            mid = f"inv:{u}->{v}"
            H.add_node(mid, gate="INV")
            H.add_edge(u, mid); H.add_edge(mid, v)
        else:
            H.add_edge(u, v)
    return H


def wl_kernel(graphs, n_iter=3):
    # function 1: normalised WL kernel matrix (1.0 = identical structure)
    from grakel import WeisfeilerLehman, VertexHistogram
    from grakel.utils import graph_from_networkx
    gk = list(graph_from_networkx([_wl_ready(G) for G in graphs], node_labels_tag="gate"))
    return WeisfeilerLehman(n_iter=n_iter, base_graph_kernel=VertexHistogram,
                            normalize=True).fit_transform(gk)


_WL_CACHE = {}


def wl_rank(target, candidates, circuits, n_iter=3):
    # function 2: [(name, similarity), ...] most similar first
    key = (id(circuits), tuple(circuits), n_iter)
    if key not in _WL_CACHE:
        names = list(circuits)
        _WL_CACHE[key] = ({n: i for i, n in enumerate(names)},
                          wl_kernel([circuits[n] for n in names], n_iter))
    idx, K = _WL_CACHE[key]
    out = [(c, float(K[idx[target], idx[c]])) for c in candidates if c != target]
    return sorted(out, key=lambda x: x[1], reverse=True)

## baseline 3 - structural
`[#nodes, #edges, depth, #PI, #PO]`, took the log so big circuits don't dominate. This is the "can the GNN cheat" check from the slides

In [8]:
def structural_vector(G):
    # function 1: log of [#nodes, #edges, depth, #PI, #PO]
    s = aig_stats(G)
    return np.log1p([s["nodes"], s["edges"], s["depth"], s["PI"], s["PO"]])


def structural_rank(target, candidates, circuits):
    # function 2: [(name, similarity), ...] most similar first
    t = structural_vector(circuits[target])
    out = [(c, 1 / (1 + float(np.linalg.norm(t - structural_vector(circuits[c])))))
           for c in candidates if c != target]
    return sorted(out, key=lambda x: x[1], reverse=True)

## eval
- Hit@1 = is the top result the same circuit
- P@k = how many of the top k are the same circuit (divided by min(k, # of variants) so 1.0 is possible)
- MRR = 1 / rank of the first correct match, averaged

In [9]:
def family(name):
    # 'c432_v3' -> 'c432'
    return name.split("_", 1)[0]


def evaluate(rank_fn, circuits, k=(5, 10), **kw):
    # rank every circuit against all the others and score the rankings
    names = list(circuits)
    hit1, rr, prec = [], [], {kk: [] for kk in k}
    for t in names:
        n_rel = sum(1 for c in names if c != t and family(c) == family(t))
        if n_rel == 0:
            continue
        hits = [family(c) == family(t) for c, _ in rank_fn(t, names, circuits, **kw)]
        hit1.append(float(hits[0]))
        rr.append(1 / (hits.index(True) + 1))
        for kk in k:
            prec[kk].append(sum(hits[:kk]) / min(kk, n_rel))
    if not hit1:
        return {}
    out = {"Hit@1": np.mean(hit1), **{f"P@{kk}": np.mean(v) for kk, v in prec.items()},
           "MRR": np.mean(rr)}
    return {m: round(float(v), 3) for m, v in out.items()}


def embedding_ranker(embeddings):
    # turns the GNN embeddings ({name: vector}) into a rank function, ranks by cosine
    E = {n: np.asarray(v, dtype=float).ravel() for n, v in embeddings.items()}

    def rank(target, candidates, circuits=None):
        out = [(c, _cosine(E[target], E[c])) for c in candidates if c != target]
        return sorted(out, key=lambda x: x[1], reverse=True)
    return rank

## temporary variants
Eesha + Jotish's variants aren't done yet so this makes some with ABC for now (balance, rewrite, refactor, dc2, etc). Each one gets CEC checked and I drop duplicates. Set `USE_STANDIN_VARIANTS = False` once theirs are in.

In [10]:
import warnings


VARIANT_SCRIPTS = {
    "v1": "balance", "v2": "rewrite", "v3": "refactor", "v4": "balance; rewrite; refactor",
    "v5": "rewrite -z; refactor -z",
    "v6": "balance; rewrite; refactor; balance; rewrite; rewrite -z; balance; refactor -z; rewrite -z; balance",
    "v7": "dc2", "v8": "if -K 4; strash", "v9": "if -K 6; strash; balance", "v10": "drw; drf",
    "v11": "if -K 3; strash; rewrite", "v12": "resub; rewrite -z",
}


def make_standin_variants(orig_dir, out_dir="standin_variants"):
    # makes up to 12 variants per circuit, CEC checks them, drops duplicates
    os.makedirs(out_dir, exist_ok=True)
    abc = lambda cmd: subprocess.run(["yosys-abc", "-c", cmd], capture_output=True, text=True).stdout

    def body(f):   # structure fingerprint: same graph = same hash, whatever the wire names are
        G = load_aig(f)
        for u, v, d in G.edges(data=True):
            d["inv"] = str(d["inverted"])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return nx.weisfeiler_lehman_graph_hash(G, node_attr="gate", edge_attr="inv", iterations=5)
    for p in sorted(glob.glob(os.path.join(orig_dir, "*.bench"))):
        c = os.path.splitext(os.path.basename(p))[0]
        base = os.path.join(out_dir, f"_{c}_strash.bench")
        abc(f'read_bench "{p}"; strash; write_bench -l "{base}"')
        seen, kept = {body(base)}, 0
        for v, script in VARIANT_SCRIPTS.items():
            out = os.path.join(out_dir, f"{c}_{v}.bench")
            abc(f'read_bench "{p}"; strash; {script}; strash; write_bench -l "{out}"')
            ok = "are equivalent" in abc(f'cec "{p}" "{out}"')
            h = body(out)
            if not ok or h in seen:
                os.remove(out)
                if not ok: print("  CEC failed, dropped:", out)
                continue
            seen.add(h); kept += 1
        os.remove(base)
        print(f"{c}: {kept} variants")
    return out_dir

In [11]:
if USE_STANDIN_VARIANTS:
    make_standin_variants(ORIG_DIR, "standin_variants")
    sources = [ORIG_DIR, "standin_variants"]
else:
    sources = [ORIG_DIR, VARIANT_DIR]

shutil.rmtree("all_bench", ignore_errors=True); os.makedirs("all_bench")
for d in sources:
    for f in glob.glob(os.path.join(d, "*.bench")):
        shutil.copy(f, "all_bench")

c1355: 10 variants


c17: 1 variants


c1908: 10 variants


c2670: 11 variants


c3540: 11 variants


...

c499: 9 variants


c5315: 11 variants


c6288: 6 variants


c7552: 11 variants


c880: 10 variants


## load everything

In [12]:
make_aigs("all_bench", "aig")
circuits = load_folder("aig")
print(len(circuits), "circuits")
pd.DataFrame({n: aig_stats(G) for n, G in circuits.items()}).T.loc[sorted(circuits, key=lambda n: (family(n), n))].head(15)

111 circuits -> aig/


111 circuits


,nodes,edges,depth,PI,PO
c1355,577,1040,26,41,32
c1355_v1,573,1032,25,41,32
c1355_v10,465,816,17,41,32
c1355_v11,517,920,23,41,32
c1355_v12,483,852,19,41,32
c1355_v2,517,920,19,41,32
c1355_v3,533,952,24,41,32
c1355_v4,497,880,18,41,32
c1355_v5,461,808,18,41,32
c1355_v6,463,812,16,41,32


## example - top 8 for c432

In [13]:
target = "c432"
names = list(circuits)
tops = {
    "Histogram": histogram_rank(target, names, circuits)[:8],
    "WL (h=3)": wl_rank(target, names, circuits)[:8],
    "Structural": structural_rank(target, names, circuits)[:8],
}
pd.DataFrame({b: [f"{n}  {s:.3f}" + ("" if family(n) == family(target) else "  (wrong)") for n, s in r]
              for b, r in tops.items()}, index=range(1, 9))

,Histogram,WL (h=3),Structural
1,c432_v1 1.000,c432_v10 0.987,c432_v3 0.988
2,c432_v9 1.000,c432_v3 0.986,c432_v10 0.932
3,c432_v3 1.000,c432_v2 0.985,c432_v2 0.911
4,c432_v10 1.000,c432_v1 0.976,c432_v5 0.778
5,c432_v2 1.000,c432_v9 0.976,c432_v12 0.754
6,c432_v12 1.000,c432_v4 0.972,c432_v1 0.665
7,c432_v5 0.999,c432_v5 0.962,c432_v9 0.665
8,c432_v4 0.999,c432_v12 0.961,c432_v4 0.632


## results

In [14]:
results = {
    "Histogram": evaluate(histogram_rank, circuits),
    "WL (h=1)": evaluate(wl_rank, circuits, n_iter=1),
    "WL (h=3)": evaluate(wl_rank, circuits, n_iter=3),
    "WL (h=5)": evaluate(wl_rank, circuits, n_iter=5),
    "Structural": evaluate(structural_rank, circuits),
}
pd.DataFrame(results).T

,Hit@1,P@5,P@10,MRR
Histogram,0.802,0.667,0.613,0.869
WL (h=1),0.910,0.883,0.805,0.950
WL (h=3),0.928,0.883,0.776,0.953
WL (h=5),0.928,0.849,0.744,0.951
Structural,0.937,0.923,0.892,0.959


## GNN vs baselines
Once the GNN works, put the embeddings in `gnn_embeddings` as `{name: vector}` and rerun. Something like:
```python
gnn_embeddings = {name: model(to_pyg(G)).detach().cpu().numpy() for name, G in circuits.items()}
```

In [15]:
gnn_embeddings = None   # {circuit_name: embedding vector}

table = dict(results)
if gnn_embeddings is not None:
    table["GNN"] = evaluate(embedding_ranker(gnn_embeddings), circuits)
pd.DataFrame(table).T

,Hit@1,P@5,P@10,MRR
Histogram,0.802,0.667,0.613,0.869
WL (h=1),0.910,0.883,0.805,0.950
WL (h=3),0.928,0.883,0.776,0.953
WL (h=5),0.928,0.849,0.744,0.951
Structural,0.937,0.923,0.892,0.959


## notes
- once everything is an AIG all 3 baselines do pretty well. structural is the best (~94% top-1) even though it's only 5 numbers. histogram ~80%, WL ~93%
- so basically on whole circuit variants, size + depth gives away which circuit it is (same thing the GNN slides said)
- the GNN needs to beat the structural row, not just histogram
- real test will probably be the cone dataset, same size circuits but different functions. this should run on it without changes